In [19]:
# ===== 팀 프로젝트 시작 셀 =====
import os, sys
from google.colab import userdata

TOKEN = userdata.get('GITHUB_TOKEN')
NAME = "yurim"
REPO_URL = "github.com/DA15-Lab/team3-ping-analytics.git"
PROJECT_DIR = "/content/team3-ping-analytics"

if not os.path.exists(PROJECT_DIR):
    !git clone -q https://{TOKEN}@{REPO_URL}
    %cd {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull -q

!git config user.email "{NAME}@example.com"
!git config user.name "{NAME}"

!pip install -q -r requirements.txt

sys.path.append("/content/team3-ping-analytics/src")
import preprocess

print(preprocess.module_test())
print("✅ 준비 완료")

/content/team3-ping-analytics
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 82.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 MB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 18.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 74.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 89.8 MB/s eta 0:00:00
✅ 모듈 설치 완료
✅ 준비 완료


In [ ]:
# ===== 작업 저장 셀 =====
%cd {PROJECT_DIR}

from google.colab import _message
import json

with open('notebooks/yurim_analysis.ipynb', 'w') as f:
    json.dump(_message.blocking_request('get_ipynb', timeout_sec=30)['ipynb'], f)

!pip freeze | grep -v "google-colab" > requirements.txt

!git add .
!git commit -m "polls_usercandidate 전처리: 중복/결측치/FK 검증 완료, 이상치 없음"
!git config pull.rebase false
!git pull
!git push

## 전처리

In [3]:
from google.colab import drive
import pandas as pd
import os

drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/00_원본 csv/polls_usercandidate.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
candidate = pd.read_csv(DATA_DIR)

In [7]:
candidate.shape

(4769609, 4)

In [8]:
candidate.head()

,id,created_at,question_piece_id,user_id
0,3088872,2023-04-28 12:27:49,998458,849444
1,3088873,2023-04-28 12:27:49,998458,849454
2,3088874,2023-04-28 12:27:49,998458,849460
3,3088875,2023-04-28 12:27:49,998458,849469
4,3088964,2023-04-28 12:28:02,998459,849446


In [9]:
candidate.dtypes

,0
id,int64
created_at,object
question_piece_id,int64
user_id,int64


In [10]:
candidate.isna().sum()

,0
id,0
created_at,0
question_piece_id,0
user_id,0


In [11]:
candidate['id'].duplicated().sum()

np.int64(0)

In [12]:
candidate.duplicated().sum()

np.int64(0)

In [13]:
# created_at 문자열에서 datetime 타입으로 변환
candidate['created_at'] = pd.to_datetime(candidate['created_at'])

In [14]:
# 가장 오래된 날짜 확인(비정상 과거 날짜가 있는지 확인)
candidate['created_at'].min()

Timestamp('2023-04-28 12:27:49')

In [15]:
# 가장 최근 날짜 확인(비정상 미래 날짜가 있는지 확인)
candidate['created_at'].max()

Timestamp('2024-05-08 01:36:18')

In [16]:
# accounts_user 불러오기(user_id 유효성 확인용)
user = pd.read_csv(os.path.join(os.path.dirname(DATA_DIR),'accounts_user.csv'))

In [17]:
# cadidate의 user_id 중 accounts_user에 없는 값이 있는지 확인
vaild_ids = set(user['id'])
(~candidate['user_id'].isin(vaild_ids)).sum()

np.int64(0)